# **Sequence to Sequence RNN Models: Language Translation**

## Objectives


 - Comprehend recurrent neural networks (RNN) architecture
 - Create an Encoder-Decoder model for a translation task
 - Train and evaluate the model
 - Create a generator for the translation task
 - Explain concepts related to Perplexity and BLEU score and use them for evaluating translations


# Background

Sequence-to-sequence (Seq2seq) models have transformed the landscape of natural language processing (NLP), enabling breakthroughs in tasks such as speech recognition, dialogue systems, and code generation. These models leverage Recurrent Neural Networks (RNNs) to map variable-length input sequences to variable-length output sequences through an encoder-decoder architecture.

## History of Sequence-to-Sequence Models

Sequence-to-sequence models emerged from the limitations of fixed-size input/output neural networks.
Researchers identified the necessity for architectures capable of processing arbitrary-length sequences, particularly for language translation tasks.
The landmark contribution of Cho et al. (2014) established the encoder-decoder framework that forms the foundation of modern seq2seq models.

Here are some main applications of seq2seq models:
- **Speech Recognition**: Converting spoken audio sequences into written text sequences.
- **Code Generation**: Producing executable code from natural language descriptions.
- **Dialogue Systems**: Generating contextually appropriate responses in conversational AI.

And many more tasks that require mapping one sequence domain to another.

## Introduction to RNNs

RNNs are a specialized class of neural networks built to handle time-dependent and sequential data.
They maintain a hidden state ($h_t$) that acts as a compressed memory of all previously seen inputs.
RNNs utilize feedback connections that propagate information across time steps during processing.

Recurrent Neural Networks (RNNs) operate on sequences and utilize previous states to influence the current state. Here's the general formulation of a simple RNN:

**Given:**
- $\mathbf{x}_t$: input vector at time step $t$
- $\mathbf{h}_{t-1}$: hidden state vector from the previous time step
- $\mathbf{W}_x$ and $\mathbf{W}_h$: weight matrices for the input and hidden state, respectively
- $\mathbf{b}$: bias vector
- $\sigma$: activation function (often a sigmoid or tanh)

The update equation for the hidden state $\mathbf{h}_t$ is as follows:

$$
\mathbf{h}_t = \sigma(\mathbf{W}_x \cdot \mathbf{x}_t + \mathbf{W}_h \cdot \mathbf{h}_{t-1} + \mathbf{b})
$$

It can be observed that the hidden state at time $t$ is jointly determined by the current input and the previous hidden state, enabling the network to accumulate contextual information over time.

For the output (if you're making a prediction at each time step):

$$
\mathbf{y}_t = \text{softmax}(\mathbf{W}_o \cdot \mathbf{h}_t + \mathbf{b}_o)
$$

**Where:**
- $\mathbf{W}_o$: weight matrix for the output
- $\mathbf{b}_o$: bias vector for the output

Depending on the specific task, an RNN cell can either emit an output from $h_t$ or pass it forward exclusively as internal memory. To better understand how this memory mechanism operates in practice, consider the following state transition diagram:

![State Transition Diagram](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Screenshot%202023-10-19%20at%2011.29.23%E2%80%AFAM.png)

The diagram illustrates a state transition system with three distinct states, represented by the prominent purple circles. Each state is uniquely identified by a value of $h$: $h = -1$, $h = 0$, and $h = 1$.

### State $h = -1$
- Remains in the same state when $x = 1$ (shown by the yellow self-loop).
- Transitions to the $h = 0$ state upon receiving input $x = -1$ (indicated by the red arrow).

### State $h = 0$
- Shifts to the $h = -1$ state when input $x = 1$ is received (indicated by the red arrow).
- Advances to the $h = 1$ state when input $x = -1$ is received (indicated by the red arrow).

### State $h = 1$
- Remains in the same state when $x = -1$ (shown by the yellow self-loop).
- Transitions back to the $h = 0$ state upon receiving input $x = 1$ (indicated by the red arrow).

---

In summary, this diagram effectively captures the dynamic behavior of an RNN's hidden state, demonstrating how the network transitions between memory states based on sequential inputs $x$. Depending on both the current state and the incoming input, the system either shifts to a new state or holds its current configuration — mirroring how an RNN retains and updates its internal memory over time.


# LSTM and Sequence-to-Sequence Architecture

## Long Short-Term Memory (LSTM)

In real-world applications, standard RNNs are rarely used in their basic form. Instead, advanced variants such as **Long Short-Term Memory (LSTM)** and **Gated Recurrent Units (GRU)** are preferred, as they effectively overcome the vanishing gradient problem that limits basic RNNs.

An LSTM cell is built around three core mechanisms known as **gates**:

- The **Input Gate** regulates the flow of new information into the cell's memory. By examining both the current input and the prior hidden state, it selectively determines which portions of the incoming data are worth retaining.

- The **Forget Gate** is responsible for clearing out outdated or irrelevant information from memory. It evaluates the current input alongside the previous hidden state to decide which stored information is no longer useful and should be discarded.

- The **Output Gate** governs what portion of the cell's memory is passed forward as output. Based on the current input and previous hidden state, it filters the memory to produce a meaningful output signal.

The fundamental strength of LSTMs lies in their dedicated memory state, which can independently retain or release information as needed. This design enables LSTMs to effectively capture long-range dependencies and preserve critical context from earlier positions in a sequence.

---

## Sequence-to-Sequence Architecture

Seq2seq models are structured around an **Encoder-Decoder** framework. The encoder processes the entire input sequence and compresses it into a compact representation known as the **context vector** ($h_t$). The decoder then takes this context vector and uses it to generate the output sequence step by step.

### How It Works — Translation Example

Consider translating the English sentence *"I love to travel"* into French *"J'adore voyager"* — a classic seq2seq task.

**Encoder Phase:**
- Words from the input sentence are fed into the encoder one at a time.
- At each step, an RNN cell takes the current word ($x_t$) and its internal memory ($h_t$), processes them together, and passes an updated context vector ($h_{t+1}$) to the next cell.
- Once the entire input sequence is consumed, the final context vector captures a compressed understanding of the whole sentence.

**Decoder Phase:**
- The context vector is handed off to the decoder.
- The decoder consists of RNN cells that generate output words one at a time.
- At each step, a decoder cell receives the previously generated word along with the updated context vector from the prior cell, and produces the next output word ($y_t$).

This encoder-decoder design allows seq2seq models to generate output sequences of **arbitrary length**, making them highly flexible for tasks like translation, summarization, and dialogue generation.


### Import Libraries

In [1]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn